# Lab 07: Latent Spaces

Uses `lab07_starter.py` unchanged. 8 by 8 digits, 1,438 training and 359 test images, pixels in [0, 1]. Every model has an 8-dimensional code and one hidden layer of 64 units on each side, and trains for 200 epochs at 2e-3 with batch 64 on seeds 0, 1 and 2 unless noted.

In [1]:
import json
import os
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import lab07_starter as S

SEEDS = [0, 1, 2]
LATENT_DIM = 8
HIDDEN_DIM = 64
EPOCHS, LEARNING_RATE, BATCH_SIZE = 200, 2e-3, 64
N_SAMPLES = 359  # every Frechet distance in this notebook uses 359 images
os.makedirs("figures", exist_ok=True)
results = {}
notebook_start = time.perf_counter()

## Run the starter once

Before writing anything I want to know the environment is fine, so I run the starter file exactly as it was handed out. A pass is the script finishing and printing a reconstruction error and a set of sample scores.

In [2]:
starter_start = time.perf_counter()
starter_run = subprocess.run([sys.executable, "lab07_starter.py"], capture_output=True, text=True, check=True)
print(starter_run.stdout)
print(f"starter took {time.perf_counter() - starter_start:.1f}s")

train (1438, 64) test (359, 64)
autoencoder test reconstruction MSE: 0.0163
feature network accuracy on real test images: 0.972
autoencoder samples from N(0, I): {'frechet': 86.242, 'precision': 0.437, 'recall': 0.794, 'nearest_train': 2.033}
saved starter_check.png

starter took 9.4s


It finished in well under a minute. The feature network gets 0.972 on the real test images, which is what every sample score below leans on.

## Load the data and train the one feature network

Every Fréchet distance, precision and recall in this lab has to come from the same feature network, otherwise the numbers can't be compared. I train it once here with the starter's defaults (seed 0) and reuse `feature_net` and `test_features` for every score.

In [3]:
Xtr, ytr, Xte, yte = S.load_task()
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))
print(f"pixel range: {Xtr.min().item():.2f} to {Xtr.max().item():.2f}")
print("train images per digit:", torch.bincount(ytr, minlength=10).tolist())
print("test images per digit: ", torch.bincount(yte, minlength=10).tolist())

feature_net = S.train_feature_net(Xtr, ytr)
test_features = S.get_features(feature_net, Xte)
with torch.no_grad():
    feature_net_accuracy = (feature_net(Xte).argmax(1) == yte).float().mean().item()
print(f"feature network test accuracy: {feature_net_accuracy:.3f}   features per image: {test_features.shape[1]}")
results["feature_net_accuracy"] = feature_net_accuracy

train (1438, 64) test (359, 64)
pixel range: 0.00 to 1.00
train images per digit: [151, 147, 141, 154, 151, 142, 137, 140, 136, 139]
test images per digit:  [27, 35, 36, 29, 30, 40, 44, 39, 38, 41]


feature network test accuracy: 0.972   features per image: 32


Same 0.972 as the starter's own run, so `train_feature_net` is deterministic and the network I score with is the one the starter checked.

## Write the VAE, the KL term and the loss

The starter has no VAE code, so this is all mine. The encoder ends in two heads, a mean and a log-variance, the code is drawn as `mean + std * noise` so the noise is the only random part, and the decoder returns logits for a Bernoulli likelihood on each pixel.

**Convention:** the reconstruction term is summed over the 64 pixels and the KL is summed over the 8 code dimensions, which gives one number per image. The loss is the mean of `reconstruction + beta * KL` over the images in the batch. I keep this in every experiment.

In [4]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=HIDDEN_DIM, d_latent=LATENT_DIM):
        super().__init__()
        self.d_latent = d_latent
        self.enc_body = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU())
        self.enc_mean = nn.Linear(d_hidden, d_latent)
        self.enc_logvar = nn.Linear(d_hidden, d_latent)
        self.dec = nn.Sequential(nn.Linear(d_latent, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_in))

    def encode(self, x):
        hidden = self.enc_body(x)
        return self.enc_mean(hidden), self.enc_logvar(hidden)

    def reparameterize(self, mean, logvar):
        noise = torch.randn_like(mean)               # the only random part, so gradients flow through mean and std
        return mean + torch.exp(0.5 * logvar) * noise

    def decode(self, z):
        return self.dec(z)                           # logits, one per pixel

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        return self.decode(z), mean, logvar

    @torch.no_grad()
    def sample(self, n, generator=None):
        z = torch.randn(n, self.d_latent, generator=generator)
        return torch.sigmoid(self.decode(z))         # pixel probabilities, in [0, 1]


def kl_to_standard_normal(mean, logvar):
    """Closed-form KL(N(mean, exp(logvar)) || N(0, I)), summed over code dimensions. One value per image."""
    return 0.5 * (mean ** 2 + torch.exp(logvar) - 1.0 - logvar).sum(dim=1)


def reconstruction_term(logits, x):
    """Negative Bernoulli log-likelihood of the pixels, summed over the 64 pixels. One value per image."""
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)


def vae_terms(model, xb):
    logits, mean, logvar = model(xb)
    return reconstruction_term(logits, xb), kl_to_standard_normal(mean, logvar)


def make_vae_loss(beta):
    def vae_loss(model, xb, yb):
        recon, kl = vae_terms(model, xb)
        return (recon + beta * kl).mean()            # mean over the batch
    return vae_loss


torch.manual_seed(0)
vae = VAE()
print(vae)
print("parameters:", sum(p.numel() for p in vae.parameters()))

VAE(
  (enc_body): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
  )
  (enc_mean): Linear(in_features=64, out_features=8, bias=True)
  (enc_logvar): Linear(in_features=64, out_features=8, bias=True)
  (dec): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
)
parameters: 9936


9,936 parameters. Same 64-unit hidden layer as the starter's `AE` on each side, the only extra is the second 8-unit head for the log-variance.

## Check 1: closed-form KL against a Monte Carlo estimate

If the closed form is right it has to agree with the brute-force version, the average of `log q(z) - log p(z)` over many draws from `q`. I use random means and log-variances (spread wide on purpose) and 200,000 draws for each row. A pass is every row within a few standard errors. I also print the closed form averaged over the 8 dimensions instead of summed, which is the mistake the handout warns about, so I can see how far off it would be.

In [5]:
torch.manual_seed(0)
check_means = 1.5 * torch.randn(6, LATENT_DIM)
check_logvars = torch.randn(6, LATENT_DIM)
closed_form_kl = kl_to_standard_normal(check_means, check_logvars)

posterior = torch.distributions.Normal(check_means, torch.exp(0.5 * check_logvars))
prior = torch.distributions.Normal(0.0, 1.0)
draws = posterior.sample((200_000,))                                      # (draws, rows, dims)
log_ratio = (posterior.log_prob(draws) - prior.log_prob(draws)).sum(-1)    # (draws, rows)
monte_carlo_kl = log_ratio.mean(0)
standard_error = log_ratio.std(0) / np.sqrt(len(draws))

print(f"{'row':>3} {'closed form':>12} {'Monte Carlo':>12} {'diff':>8} {'diff / SE':>9} {'averaged (wrong)':>17}")
for row in range(len(check_means)):
    diff = (closed_form_kl[row] - monte_carlo_kl[row]).item()
    print(f"{row:>3} {closed_form_kl[row]:12.4f} {monte_carlo_kl[row]:12.4f} {diff:8.4f} "
          f"{diff / standard_error[row].item():9.2f} {closed_form_kl[row].item() / LATENT_DIM:17.4f}")
max_z = ((closed_form_kl - monte_carlo_kl).abs() / standard_error).max().item()
print(f"largest |diff| in standard errors: {max_z:.2f}")
results["check_kl"] = {"closed_form": closed_form_kl.tolist(), "monte_carlo": monte_carlo_kl.tolist(),
                       "standard_error": standard_error.tolist()}

row  closed form  Monte Carlo     diff diff / SE  averaged (wrong)
  0      10.6361      10.6268   0.0092      0.88            1.3295
  1      21.9298      21.9343  -0.0045     -0.09            2.7412
  2      20.1421      20.1374   0.0047      0.16            2.5178
  3       9.3738       9.3593   0.0145      1.21            1.1717
  4       2.6726       2.6702   0.0024      0.37            0.3341
  5      16.7566      16.7472   0.0095      0.48            2.0946
largest |diff| in standard errors: 1.21


The closed form matches the Monte Carlo estimate on all six rows, the worst one is 1.21 standard errors off, which is just sampling noise. The last column is what I'd get by averaging over the 8 dimensions instead of summing: every value is exactly one eighth of the real KL, so a model trained on it would be at beta 1/8 without anything looking wrong.

## Check 2: does the reconstruction gradient reach the encoder?

The point of the trick is that the sample is a differentiable function of the mean and std. To test it I take one batch of 64 training images, use only the reconstruction term, and read the gradient norm on the encoder's mean layer. Then I do the same with the sample detached, and with the sampling wrapped in `torch.no_grad`, which are the two silent mistakes. A pass is a positive norm with the trick and exactly 0 for both broken versions, while the decoder still gets a gradient in all three (that's why the loss would still fall).

In [6]:
def gradient_norms(model, xb, sampling, include_kl=False):
    model.zero_grad()
    mean, logvar = model.encode(xb)
    if sampling == "trick":
        z = model.reparameterize(mean, logvar)
    elif sampling == "detached":
        z = model.reparameterize(mean, logvar).detach()
    elif sampling == "no_grad":
        with torch.no_grad():
            z = model.reparameterize(mean, logvar)
    loss = reconstruction_term(model.decode(z), xb).mean()
    if include_kl:
        loss = loss + kl_to_standard_normal(mean, logvar).mean()
    if loss.requires_grad:
        loss.backward()

    def norm(layer):
        return 0.0 if layer.weight.grad is None else layer.weight.grad.norm().item()
    return {"encoder mean": norm(model.enc_mean), "encoder logvar": norm(model.enc_logvar),
            "decoder output": norm(model.dec[-1])}


torch.manual_seed(0)
vae = VAE()
batch = Xtr[:64]
gradient_check = {}
for sampling, include_kl in [("trick", False), ("detached", False), ("no_grad", False), ("detached", True)]:
    torch.manual_seed(1)
    label = f"{sampling}, {'recon + KL' if include_kl else 'recon only'}"
    gradient_check[label] = gradient_norms(vae, batch, sampling, include_kl)
    print(f"{label:<22}", "  ".join(f"{k}: {v:.6f}" for k, v in gradient_check[label].items()))
results["check_gradients"] = gradient_check

trick, recon only      encoder mean: 0.797573  encoder logvar: 0.301622  decoder output: 5.671147
detached, recon only   encoder mean: 0.000000  encoder logvar: 0.000000  decoder output: 5.671147
no_grad, recon only    encoder mean: 0.000000  encoder logvar: 0.000000  decoder output: 5.671147
detached, recon + KL   encoder mean: 0.266565  encoder logvar: 0.262317  decoder output: 5.671147


**With the trick:** the reconstruction term alone sends 0.80 to the encoder's mean layer and 0.30 to the log-variance layer.
#
# **Detached or `no_grad`:** both encoder norms are exactly 0.000000, while the decoder gets the same 5.67 in every case. So the decoder keeps learning and the encoder is cut off from the reconstruction.
#
# **Detached plus KL:** the encoder does get a gradient again (0.27), but all of it comes from the KL, and the KL only pulls the code towards the prior.

To see what the detached mistake does over a whole run, I train one VAE at beta 1 with the sample detached and compare it with the correct one on the same seed. Both losses should fall, so the loss curve alone wouldn't catch it. What should give it away is reconstruction from the posterior mean.

In [7]:
class DetachedSampleVAE(VAE):
    def reparameterize(self, mean, logvar):
        return super().reparameterize(mean, logvar).detach()


@torch.no_grad()
def vae_reconstruction_mse(model, X):
    mean, _ = model.encode(X)
    return ((torch.sigmoid(model.decode(mean)) - X) ** 2).mean().item()


detach_demo = {}
for name, model_class in [("trick", VAE), ("detached", DetachedSampleVAE)]:
    torch.manual_seed(0)
    model = model_class()
    history = S.train(model, make_vae_loss(1.0), Xtr, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=0)
    with torch.no_grad():
        recon, kl = vae_terms(model, Xte)
    detach_demo[name] = {"first_loss": history[0], "last_loss": history[-1], "test_kl": kl.mean().item(),
                         "test_mse_from_mean": vae_reconstruction_mse(model, Xte)}
    print(f"{name:<9} loss {history[0]:.2f} -> {history[-1]:.2f}   test KL {kl.mean():.3f}   "
          f"test MSE from posterior mean {detach_demo[name]['test_mse_from_mean']:.4f}")
print(f"MSE of predicting the mean training image: {((Xtr.mean(0) - Xte) ** 2).mean():.4f}")
results["check_detached_training"] = detach_demo

trick     loss 38.84 -> 23.95   test KL 3.478   test MSE from posterior mean 0.0259


detached  loss 39.65 -> 27.10   test KL 0.000   test MSE from posterior mean 0.0738
MSE of predicting the mean training image: 0.0738


Both losses fall (38.8 to 24.0 with the trick, 39.7 to 27.1 detached), so the training curve alone would look fine. The detached model's KL went to 0.000, which means every posterior is the prior, and its test MSE from the posterior mean is 0.0738, the same as predicting the mean training image for everything. Its decoder learned to output the average digit and nothing else.
#
# The correct VAE only has a KL of 3.5 nats on the test images. That's a lot less information in the code than I expected at beta 1, and I want to keep an eye on it in the comparison with the autoencoder.

## Check 3: the loss at beta 1 is the single-sample negative ELBO

The ELBO for one image is `E_q[log p(x | z)] - KL(q(z | x) || p(z))`, and with one draw of `z` the first part becomes `log p(x | z)` at that draw. I rebuild that with `torch.distributions` (a Bernoulli on the logits and the library's own Normal KL), using the same draw as my loss by resetting the seed, and compare it with my loss at beta 1. A pass is a difference at float precision. I also average the ELBO's sampled form, `log p(x | z) + log p(z) - log q(z | x)`, over 2,000 draws per image, and my loss over the same number of draws. Those two should agree up to Monte Carlo noise.

In [8]:
torch.manual_seed(0)
vae = VAE()
S.train(vae, make_vae_loss(1.0), Xtr, epochs=20, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=0)  # a partly trained model
check_batch = Xte[:64]

with torch.no_grad():
    torch.manual_seed(5)
    my_loss = make_vae_loss(1.0)(vae, check_batch, None).item()

    torch.manual_seed(5)
    mean, logvar = vae.encode(check_batch)
    std = torch.exp(0.5 * logvar)
    z = mean + std * torch.randn_like(mean)
    log_likelihood = torch.distributions.Bernoulli(logits=vae.decode(z), validate_args=False).log_prob(check_batch).sum(1)
    library_kl = torch.distributions.kl_divergence(torch.distributions.Normal(mean, std),
                                                   torch.distributions.Normal(0.0, 1.0)).sum(1)
    negative_elbo = -(log_likelihood - library_kl).mean().item()
print(f"my loss at beta 1:        {my_loss:.6f}")
print(f"-ELBO, torch.distributions: {negative_elbo:.6f}   difference {abs(my_loss - negative_elbo):.2e}")

with torch.no_grad():
    n_draws = 2000
    posterior = torch.distributions.Normal(mean, std)
    z_draws = posterior.sample((n_draws,))                                              # (draws, 64, 8)
    log_px = torch.distributions.Bernoulli(logits=vae.decode(z_draws), validate_args=False).log_prob(check_batch).sum(-1)
    sampled_elbo = log_px + prior.log_prob(z_draws).sum(-1) - posterior.log_prob(z_draws).sum(-1)
    per_draw_loss = torch.stack([make_vae_loss(1.0)(vae, check_batch, None) for _ in range(n_draws)])
print(f"over {n_draws} draws: my loss {per_draw_loss.mean():.4f} +/- {per_draw_loss.std() / np.sqrt(n_draws):.4f}   "
      f"-ELBO (sampled form) {-sampled_elbo.mean(1).mean():.4f} +/- {sampled_elbo.mean(1).std() / np.sqrt(n_draws):.4f}")
results["check_elbo"] = {"my_loss": my_loss, "negative_elbo": negative_elbo}

my loss at beta 1:        25.414011
-ELBO, torch.distributions: 25.414011   difference 0.00e+00


over 2000 draws: my loss 25.2211 +/- 0.0046   -ELBO (sampled form) 25.2142 +/- 0.0020


With the same draw my loss and the `torch.distributions` version are identical (difference 0.00e+00). Over 2,000 draws my loss averages 25.221 and the sampled form of the ELBO averages 25.214, 0.007 apart, which is about 1.4 combined standard errors. So at beta 1 the loss is the negative ELBO, and the closed-form KL just removes the noise of sampling it.